# Chapter 06 — MLOps & Deployment

Di chapter 05 kamu sudah berhasil melatih model deep learning. Tapi coba tanyakan ini: model itu sekarang di mana? Apakah masih hidup di notebook Jupyter kamu? Kalau besok ada rekan tim yang ingin pakai, apakah dia bisa? Kalau enam bulan lagi data berubah, siapa yang akan melatih ulang? Di sinilah MLOps masuk.

MLOps singkatan dari **Machine Learning Operations**. Analoginya: bayangkan kamu punya resep masakan yang sudah jadi. Resep itu sebanding dengan model ML — dia "tahu" cara kerja, bisa diuji, bisa dinilai enaknya. Tapi selama resep itu cuma nempel di kulkas rumah kamu, dia tidak akan pernah disajikan di restoran. **MLOps adalah semua proses yang dibutuhkan supaya "resep" model itu bisa keluar dari notebook, masuk ke dapur production, dan tetap terjaga kualitasnya dari waktu ke waktu.**

Tiga pilar MLOps yang akan kita sentuh di chapter ini: **persistence** (cara simpan model ke file), **serving** (cara mempublikasikan model lewat API), dan **monitoring** (cara tahu kapan model mulai "basi" karena data berubah). Kita akan praktik pakai library yang umum di industri: `joblib`, `FastAPI`, `Docker` (dikonsepkan), dan `logging` dari Python standar.

Sepanjang chapter ini, kode kita akan sengaja singkat dan modular — bukan untuk jadi production-ready, tapi untuk menunjukkan **pola** yang dipakai di production. Setelah paham polanya, kamu bisa ganti library, ganti framework, ganti platform cloud platform; intinya tetap sama.

Prasyarat: chapter 01 (Python dasar, fungsi, modul), chapter 02 (NumPy, Pandas), chapter 03 (train/test split, akurasi), chapter 04 (pipeline, joblib sedikit), chapter 05 (model deep learning). Kita tidak akan pakai GPU ataulayanan cloud — semua bisa jalan di laptop kamu.


## Library yang Dipakai di Chapter Ini

Mayoritas kode kita pakai library yang **sudah ada di environment chapter sebelumnya** (`numpy`, `pandas`, `scikit-learn`, `joblib`). Yang baru cuma `fastapi` dan `uvicorn` untuk bikin API, plus `pydantic` (biasanya sudah ikut FastAPI). Untuk Docker, kita tidak akan benar-benar menjalankan image — cukup tulis `Dockerfile` di markdown saja supaya konsepnya jelas.

Library `logging` itu bawaan Python, jadi tidak perlu install. `hashlib` juga bawaan Python — kita pakai untuk bikin "sidik jari" dataset (dataset hash) sebagai bagian dari versioning. Kalau `fastapi` belum terinstall di environment kamu, buka terminal dan jalankan: `pip install fastapi uvicorn pydantic`. Tapi tenang, semua code cell utama di notebook ini **tetap bisa jalan tanpa fastapi** — kita akan handle import di dalam cell supaya tidak mengganggu eksperimen yang lain.

Mari kita setup environment dulu.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import hashlib
import logging
import warnings
from datetime import datetime

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')
np.random.seed(42)

print("Environment siap. Versi library utama:")
print(f"  numpy  : {np.__version__}")
print(f"  pandas : {pd.__version__}")

Cell di atas mengimpor library standar. `joblib` adalah saudara `pickle` yang dioptimasi untuk objek yang mengandung array numpy besar (alias hampir semua model scikit-learn). `hashlib` kita pakai untuk bikin sidik jari (fingerprint) dataset — string pendek yang unik untuk tiap versi data. `logging` adalah modul Python bawaan untuk menulis catatan waktu ke file atau console.

Perhatikan `np.random.seed(42)` lagi — di chapter MLOps, reproducibility itu **lebih kritis** dari chapter sebelumnya. Kalau eksperimen kamu bisa diulang, orang lain bisa memverifikasi hasilnya. Ini salah satu pilar MLOps: **reproducibility**.

---

# Section 1 — Apa Itu MLOps & Kenapa Perlu

Bayangkan dua skenario. **Skenario pertama**: tim kamu deploy API web biasa. Kode baru naik, dites, di-push ke production, selesai. Kode yang sudah jalan di server bulan lalu masih jalan minggu ini — selama tidak ada bug, tidak ada yang berubah. **Skenario kedua**: tim kamu deploy model ML. Bulan lalu akurasinya 92%. Bulan ini, tanpa kode berubah sama sekali, akurasinya turun jadi 78%. Apa yang terjadi? Bukan bug. Bukan server down. Tapi **dunia berubah** — data baru yang masuk punya distribusi berbeda dari data training. Model kamu jadi "basi" tanpa kamu sentuh sama sekali.

Inilah bedanya software biasa dengan software ML. Software biasa deterministik: input A selalu menghasilkan output A. Software ML statistik: model belajar pola dari data training, lalu apply ke data baru. Kalau data bergeser, hasil bergeser. **MLOps adalah disiplin untuk mengelola sifat "bisa basi" ini secara sistematis**.

Tiga konsep inti MLOps yang akan menemani kita di chapter ini. **Pertama, reproducibility**: kalau orang lain menjalankan kode kamu dengan data yang sama, dia harus dapat hasil yang persis sama. Makanya `random_seed` itu bukan formalitas — itu kebutuhan produksi. **Kedua, versioning**: bukan cuma kode yang perlu di-version, model dan dataset juga. Model v1 dilatih 1 Januari dengan akurasi 92%. Model v2 dilatih 1 Juni dengan akurasi 93%. Kamu harus bisa membedakan keduanya. **Ketiga, monitoring**: setelah model jalan di production, kamu harus lihat prediksinya — apakah masuk akal, apakah ada pola error, apakah ada data yang tidak mirip data training.

Tanpa tiga hal ini, model kamu pada akhirnya akan jadi "mainan" — jalan di notebook, hilang di iterasi berikutnya.

Sekarang coba kita lihat demo kecil bagaimana model bisa "turun performanya" hanya karena data berubah — tanpa kode disentuh sama sekali. Ketik kode berikut di cell bawah dan jalankan.

In [ ]:
# Skenario: model dilatih di "dunia A", lalu dites di "dunia B"
np.random.seed(42)
dunia_A = np.random.normal(loc=0, scale=1, size=500)
dunia_B = np.random.normal(loc=2, scale=1, size=500)  # mean bergeser

# Threshold sederhana: "positif" kalau > 0
threshold = 0.0
akurasi_A = (dunia_A > threshold).mean()
akurasi_B = (dunia_B > threshold).mean()
print(f"Akurasi threshold=0 di dunia_A: {akurasi_A:.2f}")
print(f"Akurasi threshold=0 di dunia_B: {akurasi_B:.2f}")
print(f"Perbedaan: {abs(akurasi_A - akurasi_B):.2f}")

Lihat outputnya: model yang sama, threshold yang sama, kode identik — tapi akurasi di `dunia_B` berbeda jauh dari `dunia_A`. Kenapa? Karena distribusi data `dunia_B` bergeser: mean-nya 2, bukan 0. Threshold 0 jadi "terlalu rendah" — banyak data baru yang sebenarnya masuk kategori positif, tapi threshold bilang negatif.

Ini analogi dunia nyata: model approval kredit kamu dilatih di data 2020 (ekonomi normal), lalu dites di data 2023 (ekonomiyang sudah pulih berbeda). Model tidak error secara kode, tapi performanya bisa turun drastis. Di sinilah monitoring dan data drift detection jadi penting.

Sekarang coba modifikasi: ubah `threshold` ke 1.5, lalu jalankan ulang. Apakah perbedaan akurasi antara `dunia_A` dan `dunia_B` mengecil atau membesar?

In [ ]:
# Modifikasi: geser threshold
threshold = 1.5
akurasi_A = (dunia_A > threshold).mean()
akurasi_B = (dunia_B > threshold).mean()
print(f"Akurasi threshold={threshold} di dunia_A: {akurasi_A:.2f}")
print(f"Akurasi threshold={threshold} di dunia_B: {akurasi_B:.2f}")

Coba amati: dengan threshold dinaikkan ke 1.5, prediksi jadi lebih "konservatif" di kedua dunia. Tapi apakah perbedaan akurasi hilang? Tidak sepenuhnya — itulah intinya. **Data drift tidak bisa dihilangkan dengan ubah threshold; yang bisa kita lakukan adalah deteksi dan mitigasi (misal: retrain dengan data baru).**

**Mini-check refleksi.** Coba jelaskan dengan bahasamu sendiri: kenapa model ML bisa "turun performa" tanpa ada baris kode yang berubah? Apa satu hal yang bisa kamu lacak di production untuk tahu kalau ini terjadi?

---

# Section 2 — Model Persistence dengan pickle & joblib

Bayangkan kamu latih model logistic regression selama 30 menit di laptop. Hasilnya disimpan di variabel `model` — variabel ini **hilang** begitu kernel Jupyter mati atau laptop restart. Kalau kamu mau pakai model itu besok, kamu harus latih ulang dari awal. Menyebalkan. **Model persistence** adalah cara menyimpan objek Python ke file di disk, supaya bisa dimuat ulang persis sama di kemudian hari — tanpa retraining.

Python punya modul bawaan `pickle` untuk ini. Dia serialisasi objek Python jadi byte stream, lalu bisa di-deserialize balik. `joblib` dari pihak ketiga melakukan hal yang sama, tapi **dioptimasi untuk objek yang mengandung array numpy besar** — alias semua model scikit-learn dan hampir semua model deep learning. Untuk model kecil bedanya tidak terasa, tapi untuk model deep learning berskala besar, `joblib` bisa 10-100x lebih cepat dan menghasilkan file lebih kecil.

Kapan pakai yang mana? Aturan praktisnya: `pickle` untuk objek Python umum (dict, list, custom class), `joblib` untuk objek yang didominasi array numpy. Di dunia nyata, kamu akan lebih sering ketik `joblib.dump(...)` daripada `pickle.dump(...)` untuk model.

Mari kita latih model iris sederhana, lalu simpan dan muat ulang pakai kedua cara. Tujuannya: verifikasi bahwa prediksi dari model yang dimuat ulang **persis sama** dengan model asli. Ketik kode di cell bawah ini dan jalankan.

In [ ]:
X, y = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=200).fit(X, y)
akurasi_asli = model.score(X, y)
print(f"Akurasi model asli: {akurasi_asli:.4f}")

# Simpan & muat ulang dengan joblib
joblib.dump(model, "model_iris.joblib")
model_loaded = joblib.load("model_iris.joblib")
akurasi_loaded = model_loaded.score(X, y)
print(f"Akurasi setelah dimuat ulang: {akurasi_loaded:.4f}")
print(f"Prediksi identik? {np.allclose(model.predict(X), model_loaded.predict(X))}")

Lihat outputnya: akurasi model asli dan model yang dimuat ulang **persis sama** sampai 4 desimal. Lebih kuat lagi, `np.allclose(...)` mengembalikan `True` — artinya prediksi per sampel juga identik secara numerik. Ini jaminan bahwa `joblib` benar-benar menyimpan **state model** (koefisien, intercept, hyperparameter), bukan cuma strukturnya.

Sekarang bandingkan dengan `pickle`. Sintaksnya mirip banget — `pickle.dump(model, file)` dan `model_loaded = pickle.load(file)`. Bedanya cuma di API dan performa. Untuk dataset iris yang cuma 150 baris × 4 fitur, perbedaannya nyaris tidak terasa. Tapi coba bayangkan model logistic regression di dataset 1 juta baris × 500 fitur — `joblib` bisa 5-10x lebih cepat dan file 30-50% lebih kecil.

Sekarang coba modifikasi: ubah `max_iter` ke 1000, simpan ke nama file `model_iris_v2.joblib`, lalu muat ulang dan bandingkan akurasi dengan model asli.

In [ ]:
# Modifikasi: model dengan iterasi lebih tinggi, simpan dengan nama beda
import pickle
model_v2 = LogisticRegression(max_iter=1000).fit(X, y)
joblib.dump(model_v2, "model_iris_v2.joblib")

# Bandingkan ukuran file
import os
size_joblib = os.path.getsize("model_iris.joblib")
print(f"joblib v1: {size_joblib} bytes")

# Simpan juga dengan pickle untuk perbandingan
with open("model_iris.pkl", "wb") as f:
    pickle.dump(model, f)
size_pickle = os.path.getsize("model_iris.pkl")
print(f"pickle   : {size_pickle} bytes")

Untuk model sekecil ini, ukuran `joblib` dan `pickle` hampir sama (kadang identik). Tapi kamu sudah lihat polanya: `joblib.dump(obj, "nama_file")` dan `model = joblib.load("nama_file")` — dua baris itu sudah cukup untuk membuat model kamu **abadi di disk**.

**Mini-check refleksi.** Tanpa buka notebook, coba tulis di kertas (atau di cell kosong) pseudocode 4 baris untuk: (1) latih model, (2) simpan ke file, (3) tutup Python, (4) buka Python baru dan muat model itu. Tantangan kecil: di langkah mana `model.score(X, y)` akan jalan tanpa retraining?

---

# Section 3 — Model Versioning & Metadata

Coba bayangkan kamu punya folder `models/` di laptop yang isinya 50 file `.joblib` dengan nama-nama seperti `model_v1.joblib`, `model_v2.joblib`, `model_baru.joblib`, `model_fix.joblib`. Sebulan dari sekarang, kamu buka folder itu dan tanya: "yang mana yang akurasinya 93%? Yang mana yang dilatih di data 2024-Q1? Yang mana yang hyperparameternya `max_iter=200`?". Kalau kamu tidak menyimpan **metadata** di samping model, semua file itu jadi tidak bisa dibedakan.

**Model versioning** bukan cuma memberi nama file. Versioning yang baik menyimpan, di samping model, semua informasi yang kamu butuhkan untuk mereproduksi model itu: kapan dilatih, di data apa, dengan hyperparameter apa, dan performanya berapa. Pola yang umum: simpan model dan metadata-nya dalam satu dictionary, lalu simpan dictionary itu sebagai JSON. Tiap versi model punya entri sendiri, dan kamu bisa lihat sejarah eksperimen kamu secara sekilas.

Untuk "sidik jari" dataset, kita pakai `hashlib.md5` — dia mengubah isi file/data jadi string 32 karakter yang **unik** untuk isi tersebut. Kalau datanya berubah sedikit saja, hash-nya berubah total. Ini cara murah dan efektif untuk melacak "apakah data yang dipakai untuk v2 ini sama dengan data untuk v1?".

Mari kita bikin registry model sederhana: dictionary yang menyimpan dua versi model iris, masing-masing dengan metadata lengkap. Ketik kode di cell bawah ini dan jalankan.

In [ ]:
X, y = load_iris(return_X_y=True)

# "Sidik jari" dataset: hash dari bytes dataset
dataset_hash = hashlib.md5(X.tobytes()).hexdigest()[:8]
print(f"Hash dataset iris: {dataset_hash}")

# Latih dua versi model
model_v1 = LogisticRegression(max_iter=200, C=1.0).fit(X, y)
model_v2 = LogisticRegression(max_iter=500, C=0.5).fit(X, y)

# Registry: dictionary of metadata + model object
registry = {
    "model_v1": {
        "model": model_v1,
        "trained_at": datetime(2024, 1, 15).isoformat(),
        "accuracy": round(accuracy_score(y, model_v1.predict(X)), 4),
        "hyperparams": {"max_iter": 200, "C": 1.0},
        "dataset_hash": dataset_hash
    },
    "model_v2": {
        "model": model_v2,
        "trained_at": datetime(2024, 6, 20).isoformat(),
        "accuracy": round(accuracy_score(y, model_v2.predict(X)), 4),
        "hyperparams": {"max_iter": 500, "C": 0.5},
        "dataset_hash": dataset_hash
    }
}

for nama, info in registry.items():
    print(f"{nama}: akurasi={info['accuracy']}, C={info['hyperparams']['C']}")

Lihat outputnya: sekarang kita punya `registry` yang berisi dua versi model, masing-masing dengan timestamp, akurasi, hyperparameter, dan hash dataset. Coba kamu bayangkan enam bulan dari sekarang: kamu bisa langsung lihat "oh, model_v2 dilatih Juni 2024 dengan C=0.5, akurasinya X" — tanpa harus retrace eksperimen kamu.

Penting: `dataset_hash` adalah cara paling sederhana untuk **versioning dataset**. Kalau kamu ganti data (misal tambah 100 sampel baru), hash-nya akan berbeda. Ini bukan pengganti DVC (Data Version Control) yang proper, tapi cukup untuk eksperimen skala kecil-menengah.

Sekarang coba modifikasi: tambah `model_v3` ke registry, kali ini pakai `C=2.0` dan timestamp hari ini, lalu print seluruh registry sebagai JSON (kita drop object model-nya dulu supaya JSON bisa serialize).

In [ ]:
# Modifikasi: tambah model_v3 dengan hyperparameter beda
model_v3 = LogisticRegression(max_iter=300, C=2.0).fit(X, y)
registry["model_v3"] = {
    "model": model_v3,
    "trained_at": datetime.now().isoformat(),
    "accuracy": round(accuracy_score(y, model_v3.predict(X)), 4),
    "hyperparams": {"max_iter": 300, "C": 2.0},
    "dataset_hash": dataset_hash
}

# Print sebagai JSON (object model kita exclude)
registry_view = {k: {kk: vv for kk, vv in v.items() if kk != "model"}
                 for k, v in registry.items()}
print(json.dumps(registry_view, indent=2, ensure_ascii=False))

Sekarang registry kamu punya tiga versi. Kalau ada pertanyaan "model mana yang terbaik?", kamu bisa langsung lihat tabel akurasi dan hyperparameter di JSON di atas. **Ini fondasi dari experiment tracking** — alat seperti MLflow dan Weights & Biases pada dasarnya adalah registry persis seperti ini, tapi dengan UI dan penyimpananpenyimpanan cloud yang lebih serius.

**Mini-check refleksi.** Bayangkan kamu deploy model_v2 ke production. Tiga bulan kemudian ada masalah: prediksi melenceng. Hal pertama yang akan kamu cek dari registry di atas? Bayangkan juga sebaliknya: dataset berubah total, dan kamu mau tahu apakah model v1 dan v2 dilatih di dataset yang sama. Field mana yang kamu pakai untuk menjawab itu?

---

# Section 4 — REST API Serving dengan FastAPI

Setelah model dilatih dan disimpan, pertanyaan berikutnya: **gimana orang lain bisa pakai model itu?** Jawabannya: lewat **API**. API (Application Programming Interface) adalah "pintu" yang bisa dipanggil dari mana saja — aplikasi mobile, website, atau script lain. Kamu tidak perlu kirim model file ke semua orang; kamu cukup kasih mereka **endpoint** (URL) yang bisa mereka hitung, dan model kamu yang bekerja di server.

**REST API** adalah gaya API yang paling umum: dia pakai HTTP (protokol yang sama dengan website), dengan aturan: GET untuk ambil data, POST untuk kirim data, dan tiap "pintu" punya URL yang jelas. Untuk serving model ML, biasanya yang kita butuhkan cuma satu: **POST /predict** — endpoint yang menerima data input, jalankan model, kembalikan prediksi.

**FastAPI** adalah framework Python modern untuk bikin REST API. Kelebihannya: sintaksnya sangat Pythonic (hampir tidak terasa bedanya dengan fungsi biasa), otomatis generate dokumentasi, dan validasi input pakai **Pydantic** (library yang memastikan data yang masuk sesuai format yang kita harapkan). Pydantic sangat penting: tanpa validasi, server bisa crash karena input yang aneh-aneh (misal string sebagai pengganti angka).

Mari kita bikin API sederhana untuk model iris. Kode di bawah ini **tidak langsung jalan sebagai server** di notebook, tapi menunjukkan struktur lengkapnya. Jika kamu save ke file `app.py` dan jalankan `uvicorn app:app --reload`, server akan hidup. Coba ketik dan pahami dulu strukturnya.

In [ ]:
try:
    from fastapi import FastAPI
    from pydantic import BaseModel
    import uvicorn
    FASTAPI_AVAILABLE = True
    print("FastAPI tersedia. Kode API di bawah ini siap dipakai.")
except ImportError:
    FASTAPI_AVAILABLE = False
    print("FastAPI belum terinstall. Install dengan: pip install fastapi uvicorn pydantic")
    print("Kode di cell ini cuma untuk dilihat polanya — tidak akan jalan sebagai server.")

Cell di atas hanya cek apakah `fastapi` terinstall. Pendekatan `try/except` ini pola yang aman di notebook — kode eksperimen kamu tidak crash cuma karena satu library tidak ada. Kalau `FASTAPI_AVAILABLE = True`, kita bisa lanjut. Kalau `False`, kita skip bagian yang butuh library itu.

Sekarang, struktur API-nya sendiri. Perhatikan tiga komponen penting. **Pertama**, `BaseModel` dari Pydantic: class `IrisInput` mendefinisikan **bentuk** data yang boleh masuk. **Kedua**, decorator `@app.post("/predict")`: ini yang mendaftarkan fungsi `predict` sebagai endpoint. **Ketiga**, fungsi `predict` itu sendiri menerima `data: IrisInput` — FastAPI otomatis validasi dan parse JSON yang masuk jadi objek Pydantic.

Mari kita lihat kode lengkapnya di cell modifikasi. Untuk latihan, coba salin kode ini ke file `app.py` di laptop kamu, lalu jalankan `uvicorn app:app --reload` di terminal.

In [ ]:
# Kode ini hanya ILLUSTRASI struktur. Untuk menjalankannya sebagai server,
# simpan ke file app.py lalu jalankan: uvicorn app:app --reload
if FASTAPI_AVAILABLE:
    app = FastAPI(title="Iris Predictor", version="1.0")
    model = joblib.load("model_iris.joblib")

    class IrisInput(BaseModel):
        sepal_length: float
        sepal_width: float
        petal_length: float
        petal_width: float

    @app.post("/predict")
    def predict(data: IrisInput):
        features = [[data.sepal_length, data.sepal_width,
                     data.petal_length, data.petal_width]]
        prediction = int(model.predict(features)[0])
        return {"predicted_class": prediction}

    print("Untuk menjalankan server, simpan cell ini ke app.py")
    print("Lalu: uvicorn app:app --reload")
    print("Test: kirim POST ke /predict dengan JSON body ber-field sepal_length dll")

Coba pahami alur request-response. **Request masuk**: client kirim JSON `{"sepal_length": 5.1, "sepal_width": 3.5, ...}` ke `POST /predict`. **Validasi**: FastAPI cek apakah semua field ada dan tipenya benar (kalau ada yang missing atau salah tipe, otomatis kembalikan error 422). **Prediksi**: model jalan, hasilnya dikembalikan sebagai JSON `{"predicted_class": 0}`. **Response keluar**: client terima jawaban.

Keuntungan besar FastAPI: dia otomatis generate dokumentasi interaktif di `http://localhost:8000/docs` — kamu bisa coba-coba endpoint langsung dari browser tanpa nulis `curl` sekali pun.

Sekarang coba modifikasi: tambahkan field `confidence` ke response. Caranya: setelah `model.predict(...)`, panggil juga `model.predict_proba(features).max()` untuk ambil probabilitas tertinggi, lalu masukkan ke dictionary response.

**Mini-check refleksi.** Coba jelaskan dengan bahasamu sendiri: kenapa kita perlu `BaseModel` dari Pydantic? Apa yang akan terjadi kalau client kirim `{"sepal_length": "bukan angka"}` ke endpoint kita?

In [ ]:
# Modifikasi kecil: ilustrasi response dengan confidence
if FASTAPI_AVAILABLE:
    # Simulasi request & response
    import random
    sample = IrisInput(sepal_length=5.1, sepal_width=3.5,
                       petal_length=1.4, petal_width=0.2)
    features = [[sample.sepal_length, sample.sepal_width,
                 sample.petal_length, sample.petal_width]]
    pred_class = int(model.predict(features)[0])
    confidence = float(model.predict_proba(features).max())
    response = {"predicted_class": pred_class, "confidence": round(confidence, 4)}
    print("Contoh response:", response)

---

# Section 5 — Dockerizing Model: Dari Notebook ke Container

Model sudah bisa dipanggil lewat API. Tapi ada satu masalah: API itu jalan di laptop kamu, pakai Python 3.11, dengan library yang terinstall di environment kamu. Kalau rekan kamu mau jalanin API yang sama di laptopnya, kemungkinan besar ada masalah: "kok `fastapi` not defined?", "kok Python 3.9?", "kok beda versi `scikit-learn`?". **Docker** menjawab masalah ini.

Bayangkan **image** Docker itu seperti **blueprint** rumah. Blueprint berisi semua informasi: pondasi, tembok, atap, instalasi listrik, dan perabotan apa saja yang ada di dalam rumah. **Container** adalah **rumah** yang dibangun dari blueprint itu. Dari satu blueprint, kamu bisa bangun banyak rumah identik di mana saja — di laptop kamu, di server temanmu, diplatform cloud platform AWS. Rumah pertama, kedua, ketiga, semuanya persis sama isinya.

Untuk model ML, "isi rumah" itu biasanya: kode API (FastAPI), model file (`.joblib`), library yang dibutuh (`fastapi`, `scikit-learn`), Python versi tertentu. Semua itu dideklarasikan di file bernama `Dockerfile`. Tools Docker baca Dockerfile, lalu build image. Image itu yang kamu distribusikan.

Di bawah ini adalah contoh `Dockerfile` untuk API iris yang kita tulis di section 4. Dockerfile bukan kode yang dijalankan — dia adalah **resep** untuk Docker. Coba pahami setiap barisnya, dan jalankan cell di bawah ini untuk print ke layar (tidak benar-benar build Docker).

In [ ]:
dockerfile_content = '''
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY model_iris.joblib .

EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''.strip()

requirements_content = '''
fastapi==0.110.0
uvicorn==0.27.1
scikit-learn==1.4.1
joblib==1.3.2
'''.strip()

print("=== Dockerfile ===")
print(dockerfile_content)
print("\n=== requirements.txt ===")
print(requirements_content)

Coba amati setiap baris Dockerfile. **`FROM python:3.11-slim`**: ini "fondasi" image. Kita mulai dari image Python 3.11 versi ringan (slim). **`WORKDIR /app`**: bikin folder `/app` di dalam container, dan semua perintah berikutnya jalan di situ. **`COPY requirements.txt .`**: salin file requirements dari laptop ke image. **`RUN pip install ...`**: install semua library. **`COPY app.py .`** dan **`COPY model_iris.joblib .`**: salin kode API dan model file. **`EXPOSE 8000`**: deklarasi bahwa container akan dengar di port 8000. **`CMD [...]`**: perintah yang dijalankan waktu container start.

Urutan baris-baris itu bukan kebetulan. Docker **cache** setiap layer, dan urutannya menentukan efisiensi build. `COPY requirements.txt` ditaruh **sebelum** `COPY app.py` karena: kalau kamu cuma ubah `app.py` (tambah fitur baru), Docker tidak perlu install ulang library — cache `pip install` masih bisa dipakai. Build jadi jauh lebih cepat.

Sekarang coba modifikasi: tambah `HEALTHCHECK` ke Dockerfile. Ini memberitahu Docker cara cek "apakah container ini masih sehat?". Formatnya: `HEALTHCHECK CMD curl -f http://localhost:8000/docs || exit 1`. Coba tambahkan baris itu sebelum `CMD [...]` dan pikirkan kenapa itu penting untuk production.

In [ ]:
# Modifikasi: Dockerfile dengan HEALTHCHECK
dockerfile_with_health = dockerfile_content.replace(
    'CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]',
    'HEALTHCHECK CMD curl -f http://localhost:8000/docs || exit 1\nCMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]'
)
print(dockerfile_with_health)

Perubahan kecil ini punya dampak besar. Tanpa `HEALTHCHECK`, kalau API kamu crash tapi proses container masih jalan (kasus yang sering terjadi), Docker tidak tahu ada masalah. Dengan `HEALTHCHECK`, Docker secara berkala cek ke endpoint `/docs` — kalau 3x berturut-turut gagal, container dianggap `unhealthy` dan orchestrator (Kubernetes, Docker Swarm) bisa restart-nya otomatis.

**Image vs container, sekali lagi**: image = blueprint, container = rumah jadi. Satu image bisa di-spawn jadi banyak container (misal: 5 container untuk handle traffic tinggi). Masing-masing container independen, tapi semuanya menjalankan kode yang sama. Inilah kenapa Docker jadi tulang punggung deployment modern.

**Mini-check refleksi.** Coba jelaskan dengan bahasamu sendiri: kenapa tidak cukup hanya copy file `app.py` dan `model_iris.joblib` ke server rekan kamu lewat USB? Apa yang akan lebih terjamin kalau pakai Docker?

---

# Section 6 — CI/CD Pipeline untuk ML

Kamu mungkin sudah dengar istilah **CI/CD** di dunia software engineering. **CI (Continuous Integration)**: setiap kali developer push kode, server otomatis menjalankan test untuk memastikan tidak ada yang rusak. **CD (Continuous Deployment)**: kalau test lolos, kode baru otomatis di-deploy ke production. Tujuannya: **kurangi human error, percepat release**.

Di dunia ML, CI/CD punya **lapis tambahan** yang software biasa tidak punya. CI/CD software biasa: test kode (apakah fungsi `add(a, b)` mengembalikan `a + b`). CI/CD ML: **test model** (apakah akurasi model baru tidak turun dari baseline). Tanpa tes model, kamu bisa deploy kode yang syntactically benar, berjalan tanpa error, tapi memberikan prediksi lebih buruk dari model sebelumnya. Ini skenario nightmare yang sering banget terjadi di production.

Konsepnya: setiap kali data training baru masuk (atau kode training berubah), **jalankan pipeline otomatis** yang: (1) latih model baru, (2) evaluasi di test set, (3) bandingkan dengan model baseline — kalau lebih baik, simpan & deploy; kalau lebih buruk, jangan deploy. Di sinilah registry dari section 3 berperan penting: model baseline ada di registry dengan metadata akurasinya.

Mari kita tulis logika **promote-or-reject** untuk model baru. Idenya: kalau akurasi model baru lebih baik dari baseline, simpan ke registry. Kalau lebih buruk, log warning dan jangan timpa. Ketik kode di cell bawah ini dan jalankan.

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model baseline (anggap ini yang sedang di-production)
baseline_model = LogisticRegression(max_iter=200, C=1.0).fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline_model.predict(X_test))
print(f"Akurasi baseline (di production): {baseline_acc:.4f}")

# Kandidat model baru dengan hyperparameter berbeda
candidate_model = LogisticRegression(max_iter=200, C=0.3).fit(X_train, y_train)
candidate_acc = accuracy_score(y_test, candidate_model.predict(X_test))
print(f"Akurasi kandidat baru:           {candidate_acc:.4f}")

# Logika promote-or-reject
if candidate_acc > baseline_acc:
    print("KANDIDAT DITERIMA: simpan & deploy")
    joblib.dump(candidate_model, "model_candidate.joblib")
else:
    print("KANDIDAT DITOLAK: akurasi tidak membaik")

Lihat outputnya: pipeline secara otomatis memutuskan apakah model baru layak naik. Di production, keputusan ini diambil oleh server (GitHub Actions, GitLab CI, Jenkins) setiap kali ada push ke branch `main` atau setiap kali data baru masuk. Yang penting: **keputusan ini objektif dan tercatat** — tidak ada debat "menurut saya model ini lebih baik" yang subjektif.

Logika promote-or-reject di atas sangat sederhana — cuma bandingkan satu angka. Di production, sering ada kriteria tambahan: akurasi harus naik minimal 0.5%, **dan** recall kelas minoritas tidak boleh turun, **dan** model file tidak boleh lebih besar dari 100MB. Logikanya bisa setingkat apa pun yang kamu butuhkan.

Sekarang coba modifikasi: ubah threshold promotion. Saat ini, model diterima hanya kalau **lebih baik** (`>`). Ganti ke **tidak lebih buruk dari baseline dikurangi toleransi 1%** (`>= baseline_acc - 0.01`). Ini lebih permisif: model diterima selama tidak turun drastis, tidak harus naik.

In [ ]:
# Modifikasi: threshold promotion yang lebih permisif
toleransi = 0.01
if candidate_acc >= baseline_acc - toleransi:
    print(f"DITERIMA (toleransi {toleransi}): simpan & deploy")
else:
    print(f"DITOLAK: turun lebih dari {toleransi:.0%} dari baseline")

Kenapa threshold seperti ini lebih sering dipakai di production? Karena kenaikan akurasi 0.1% itu **noise** — bisa jadi cuma kebetulan dari random seed. Yang lebih penting adalah **menjaga degradasi** — pastikan model baru tidak tiba-tiba drop. Toleransi 0.5-1% adalah angka yang umum.

Tentu saja, semua ini bukan keputusan akhir. Profesional ML yang baik akan menambah: (1) test pada **slice** data yang berbeda (misal per kelas, per region), (2) cek **fairness metrics** (apakah model diskriminatif ke grup tertentu), (3) **A/B test** di production: jalankan model lama dan baru secara paralel selama 1-2 minggu, bandingkan metrik bisnis. Tapi untuk fondasi, logika promote-or-reject di cell sebelumnya sudah cukup.

**Mini-check refleksi.** Tanpa buka catatan, jelaskan perbedaan utama CI/CD software biasa dengan CI/CD ML. Beri satu contoh "test" yang hanya relevan untuk ML (tidak ada di software biasa).

---

# Section 7 — Monitoring & Logging

Model sudah di-deploy. Andaikan API jalan lancar, tidak ada error 500, response time stabil. Apakah semuanya beres? **Belum.** Ada tiga hal yang perlu kamu pantau terus-menerus setelah deploy. **Pertama, traffic**: berapa request per menit? Apakah ada lonjakan tiba-tiba? **Kedua, latency**: berapa lama rata-rata satu request diproses? Apakah mulai melambat? **Ketiga, distribusi input & output**: data yang masuk model, apakah masih mirip dengan data training? Apakah proporsi kelas output masih sama?

Logging adalah cara termurah dan paling universal untuk mulai. Modul `logging` Python (bawaan, tidak perlu install) bisa menulis catatan ke console, ke file, atau bahkan ke remote server. Kamu bisa log setiap prediksi: timestamp, input, output, latency. Ini **goldmine** kalau ada masalah — kamu bisa mundur ke belakang dan lihat "oh, sejak 3 hari yang lalu proporsi kelas 1 mulai naik".

Yang penting: **logging bukan `print()`**. Print() bisa hilang di output cell, tidak ada timestamp, tidak ada level (info vs warning vs error), dan tidak bisa diarahkan ke file. Logging punya semua itu.

Mari kita setup logger sederhana, lalu simulasikan monitoring: catat 50 prediksi, hitung rata-rata latency dan proporsi kelas, lalu cek apakah ada pola anomali. Ketik kode di cell bawah dan jalankan.

In [ ]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s',
                    datefmt='%H:%M:%S')
logger = logging.getLogger(__name__)

# Simulasi 50 request prediksi
model = joblib.load("model_iris.joblib")
latencies_ms = []
pred_counts = {0: 0, 1: 0, 2: 0}

for i in range(50):
    sample = X[i % len(X):i % len(X) + 1]
    t0 = datetime.now()
    pred = int(model.predict(sample)[0])
    latency = (datetime.now() - t0).total_seconds() * 1000

    latencies_ms.append(latency)
    pred_counts[pred] += 1
    logger.info(f"request#{i+1} latency={latency:.2f}ms pred={pred}")

logger.info(f"Rata-rata latency: {np.mean(latencies_ms):.3f} ms")
logger.info(f"Distribusi prediksi: {pred_counts}")

Lihat outputnya: tiap baris log punya timestamp, level (INFO), dan pesan terstruktur. Ini jauh lebih readable dan searchable dibanding deretan `print()`. Coba bayangkan: kalau ada insiden, kamu bisa grep `WARNING` atau `ERROR` dari log file dan langsung tahu kapan masalah mulai.

Tiga angka penting yang harus kamu pantau dari output di atas. **Rata-rata latency** `~0.05-0.5 ms` untuk model iris — sangat cepat, tidak perlu optimasi. Untuk model yang lebih besar, kalau latency mulai naik dari 5ms ke 50ms, ada tanda ada masalah. **Distribusi prediksi**: di log, proporsi kelas harusnya mirip dengan proporsi di data training. Kalau tiba-tiba kelas 2 melonjak dari 30% ke 70%, patut dicurigai. **Error logs**: kalau ada prediksi gagal atau input tidak valid, harusnya muncul sebagai WARNING atau ERROR, bukan INFO.

Sekarang coba modifikasi: ubah level logging ke `WARNING` dan jalankan ulang. Apa yang berubah di output? Lalu coba trigger satu warning (misal prediksi dengan input yang salah shape) dan lihat log-nya.

In [ ]:
# Modifikasi: log level lebih ketat + trigger warning
logger.setLevel(logging.WARNING)
logger.warning("Prediksi dengan input anomali terdeteksi: shape tidak sesuai")
logger.info("Ini tidak akan muncul karena level INFO lebih rendah dari WARNING")

Coba amati: baris `logger.warning(...)` muncul, tapi `logger.info(...)` tidak. Kenapa? Karena `setLevel(logging.WARNING)` artinya hanya log level `WARNING` ke atas yang lewat. Ini berguna untuk production: saat pengembangan (development), pakai `DEBUG` atau `INFO` untuk lihat semua; saat production, naikkan ke `WARNING` supaya log file tidak penuh dengan detail yang tidak perlu.

Prinsip umum: **log secukupnya, jangan berlebihan**. Log setiap prediksi di notebook boleh, tapi di production yang handle 1000 request per detik itu akan jadi 86 juta baris log per hari. Pilih momen yang benar-benar penting: error, warning, dan beberapa metric agregat per menit (count request, avg latency, distribusi output). Sisanya bisa di-sample.

**Mini-check refleksi.** Sebutkan tiga hal yang akan kamu log untuk model production. Untuk masing-masing, jelaskan kenapa itu penting dan "nilai normal"-nya kira-kira berapa. Contoh sudah diberikan untuk latency — kamu tinggal tambah dua lagi.

---

# Section 8 — Data Drift Detection

Dari semua hal di MLOps, **data drift** mungkin yang paling licik. Kodenya jalan, API tidak error, latency normal — tapi performa model di production turun drastis. Penyebabnya: distribusi data production bergeser dari data training. Contoh konkret: model deteksi fraud kamu dilatih di data 2023 (sebagian besar transaksi via web), lalu di 2024 transaksi pindah ke mobile dan karakteristiknya beda.

Drift bisa terjadi di **input** (fitur X berubah), **output** (target y berubah proporsinya), atau **konseptual** (hubungan X→y itu sendiri berubah — paling sulit dideteksi). Untuk drift input dan output, kita bisa pakai statistik sederhana: bandingkan mean, standar deviasi, dan distribusi kuantil data training vs data production. Kalau perbedaannya signifikan, **ada drift**.

Cara paling sederhana: **Population Stability Index (PSI)** — metrik yang umum di industri. PSI = 0 artinya dua distribusi identik; PSI < 0.1 artinya tidak ada drift signifikan; PSI 0.1-0.2 artinya drift moderat; PSI > 0.2 artinya drift besar. Untuk menghitungnya, kamu perlu membagi data ke dalam bin (misal 10 bin kuantil), lalu hitung proporsi tiap bin di kedua dataset.

Mari kita buat deteksi drift sederhana untuk fitur numerik. Idenya: kita punya `data_train` (representasi data training), lalu `data_prod` (simulasi data production). Kita hitung PSI-nya dan lihat apakah ada drift. Ketik kode di cell bawah ini dan jalankan.

In [ ]:
def calculate_psi(train, prod, bins=10):
    """Hitung Population Stability Index antara dua distribusi."""
    quantiles = np.quantile(train, np.linspace(0, 1, bins + 1))
    quantiles[0] = -np.inf
    quantiles[-1] = np.inf

    train_counts, _ = np.histogram(train, bins=quantiles)
    prod_counts, _ = np.histogram(prod, bins=quantiles)

    train_pct = (train_counts + 1) / (len(train) + bins)  # smoothing
    prod_pct = (prod_counts + 1) / (len(prod) + bins)

    psi = np.sum((prod_pct - train_pct) * np.log(prod_pct / train_pct))
    return psi

# Skenario 1: data production identik dengan training
np.random.seed(42)
data_train = np.random.normal(loc=50, scale=10, size=1000)
data_prod_sama = np.random.normal(loc=50, scale=10, size=1000)
psi_sama = calculate_psi(data_train, data_prod_sama)

# Skenario 2: data production bergeser (mean naik ke 55)
data_prod_drift = np.random.normal(loc=55, scale=10, size=1000)
psi_drift = calculate_psi(data_train, data_prod_drift)

print(f"PSI identik:        {psi_sama:.4f} (< 0.1 = tidak ada drift)")
print(f"PSI drift (mean+5): {psi_drift:.4f} (> 0.2 = drift besar)")

Lihat outputnya: dua skenario, dua hasil sangat berbeda. **Skenario 1** (data production identik dengan training) menghasilkan PSI ~0.02-0.05 — jauh di bawah threshold 0.1, artinya tidak ada drift. **Skenario 2** (mean production bergeser dari 50 ke 55, hanya 10% pergeseran) sudah menghasilkan PSI di atas 0.1 — artinya drift terdeteksi.

Kode `calculate_psi` ini punya beberapa detail penting. **Pertama**, `bins=10` artinya kita bagi distribusi jadi 10 bagian (desil). Lebih banyak bin = deteksi lebih halus, tapi kurang robust untuk sample kecil. **Kedua**, `+1` di numerator dan `+bins` di denominator adalah **smoothing** — mencegah `log(0)` yang undefined kalau ada bin kosong. **Ketiga**, hasil `np.sum(...)` adalah jumlah kontribusi per bin — bin yang proporsinya berubah paling banyak akan kontribusinya paling besar.

Sekarang coba modifikasi: buat skenario ketiga dengan pergeseran lebih kecil (mean naik dari 50 ke 52) dan hitung PSI-nya. Apakah drift sudah terdeteksi di threshold 0.1, atau masih terlalu kecil?

In [ ]:
# Modifikasi: pergeseran mean yang lebih halus (50 -> 52)
data_prod_kecil = np.random.normal(loc=52, scale=10, size=1000)
psi_kecil = calculate_psi(data_train, data_prod_kecil)
print(f"PSI drift kecil (mean+2): {psi_kecil:.4f}")
if psi_kecil < 0.1:
    print("-> Belum cukup untuk dianggap drift signifikan")
elif psi_kecil < 0.2:
    print("-> Drift moderat, perlu waspada")
else:
    print("-> Drift besar, perlu retraining segera")

Coba amati: pergeseran mean dari 50 ke 52 (hanya 4% dari standar deviasi) menghasilkan PSI berapa? Biasanya di kisaran 0.03-0.08 — masih di bawah threshold 0.1, artinya belum dianggap drift signifikan. Ini masuk akal: pergeseran sekecil itu memang tidak akan mengubah performa model secara material.

**Di production, threshold 0.1 sering disesuaikan dengan konteks.** Model approval kredit yang impact-nya ke jutaan orang? Threshold lebih ketat, 0.05. Model rekomendasi film? Threshold bisa lebih longgar, 0.15. Yang penting bukan angka persisnya, tapi **kamu punya baseline dan alert-nya jelas**.

Kalau drift terdeteksi (PSI > threshold), langkah selanjutnya biasanya: (1) cek apakah data production memang valid (bukan bug ingestion), (2) kalau valid, **retrain model** dengan data terbaru, (3) validasi model baru di test set yang representatif, (4) deploy seperti pipeline CI/CD di section 6. Loop ini — pantau, deteksi, retrain, deploy — adalah **siklus hidup ML di production**. Bukan sekali jalan.

**Mini-check refleksi.** Bayangkan kamu deploy model prediksi harga rumah. Tahun pertama, mean harga di training data 800 juta. Tahun kedua, mean harga production 950 juta. Apakah itu drift? Hitung kira-kira PSI-nya (kamu tidak perlu hitung persis, kira-kira saja: apakah pergeseran 18% dari mean akan menembus threshold 0.1?). Apa yang akan kamu lakukan?

---

# Penutup Chapter 06

Kita sudah menyentuh delapan pilar MLOps: konsep, persistence, versioning, serving via API, containerization, CI/CD, monitoring, dan drift detection. Pola-pola ini adalah **bahasa umum** yang dipakai di industri, dari startup kecil sampai perusahaan teknologi besar. Library dan framework bisa beda (MLflow vs Weights & Biases, Airflow vs Prefect, Kubernetes vs ECS), tapi konsepnya identik.

Kalau kamu serius masuk ke dunia ML engineering, tiga skill yang akan paling membedakan kamu: (1) **membuat API yang clean dan terdokumentasi** (FastAPI, Pydantic), (2) **menulis pipeline yang reproducible** (Makefile, Docker, GitHub Actions), dan (3) **membangun sistem monitoring** yang bisa mendeteksi masalah sebelum user complain. Ketiganya kita sudah lihat dasar-dasarnya di chapter ini.

Chapter selanjutnya (kalau ada) akan melompat ke topik big data: Spark, distributed computing, dan stream processing. Atau kembali ke fondasi matematika yang lebih dalam. Apapun arahnya, pola MLOps yang sudah kita pelajari akan selalu relevan: dunia berubah, dan model yang kita buat harus bisa mengikuti perubahan itu dengan cara yang terukur dan otomatis.

Selamat — kamu sudah menyelesaikan materi MLOps dasar. Langkah selanjutnya? Coba bikin API iris yang kita tulis di section 4, simpan ke file `app.py`, lalu **benar-benar jalankan** di laptop kamu: `uvicorn app:app --reload`. Rasakan sensasi punya "model beneran" yang bisa dipanggil orang lain lewat internet. Itu permulaan yang bagus.